# CUDA chunkwise kernels: GDN / KDA versus fused SDPA

This notebook replaces the **reference-path throughput claim** with an operator-level CUDA experiment. It uses FLA's chunkwise GDN/KDA kernels for prefill, FLA's fused recurrent kernels for one-token decode, and PyTorch SDPA as the full-attention control.

It does **not** retrain the small GPT model or change the previously measured PPL/cache results. First verify numerical agreement with the exact recurrence; only then use the timing tables in the research report.

**Colab:** select `Runtime → Change runtime type → T4 GPU` (or better), then run cells top to bottom.

In [ ]:
# FLA v0.5.1 has CUDA, chunked GDN/KDA, and decode kernels.
# If pip upgrades torch in this Colab runtime, restart the session once, then resume below.
!pip -q install -U 'flash-linear-attention[cuda]==0.5.1' pandas
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader

In [ ]:
import platform
import time
from dataclasses import dataclass

import pandas as pd
import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), 'Enable a GPU runtime in Colab, then rerun.'
from fla.ops.gated_delta_rule import chunk_gated_delta_rule, fused_recurrent_gdn
from fla.ops.kda import chunk_kda, fused_recurrent_kda

DEVICE = torch.device('cuda')
GPU_NAME = torch.cuda.get_device_name()
CAPABILITY = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if CAPABILITY[0] >= 8 else torch.float16
torch.manual_seed(2026)
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')
print({'torch': torch.__version__, 'gpu': GPU_NAME, 'capability': CAPABILITY, 'dtype': str(DTYPE)})

## Matched synthetic operator inputs

The shapes match the 4-layer prototype's token mixer: `hidden = 128`, `H = 4`, `head_dim = 32`. Q/K are L2-normalized before both paths, `g` is already log-decay, `beta` is already sigmoid-gated, and the recurrent state accumulates in FP32. This makes the FLA call algebraically comparable to `research/linear_attention.py`.

In [ ]:
@dataclass(frozen=True)
class Shape:
    batch: int = 8
    heads: int = 4
    head_dim: int = 32

SHAPE = Shape()

def make_inputs(tokens, *, channelwise, batch=SHAPE.batch):
    b, h, d = batch, SHAPE.heads, SHAPE.head_dim
    q = F.normalize(torch.randn(b, tokens, h, d, device=DEVICE), dim=-1).to(DTYPE)
    k = F.normalize(torch.randn(b, tokens, h, d, device=DEVICE), dim=-1).to(DTYPE)
    v = torch.randn(b, tokens, h, d, device=DEVICE, dtype=DTYPE)
    gate_shape = (b, tokens, h, d) if channelwise else (b, tokens, h)
    # Strictly negative log-decay, matching the reference rule.
    g = (-F.softplus(torch.randn(gate_shape, device=DEVICE)) - 0.05).to(DTYPE)
    beta = torch.sigmoid(torch.randn(b, tokens, h, device=DEVICE)).to(DTYPE)
    state = torch.zeros(b, h, d, d, device=DEVICE, dtype=torch.float32)
    return q, k, v, g, beta, state

def reference_scan(q, k, v, g, beta, *, channelwise, initial_state):
    """Exact recurrence used only for correctness and short reference timings."""
    q, k, v, g, beta = (x.float() for x in (q, k, v, g, beta))
    state = initial_state.float()
    outputs = []
    for token in range(q.shape[1]):
        decay = g[:, token].exp()
        decay = decay.unsqueeze(-1) if channelwise else decay.unsqueeze(-1).unsqueeze(-1)
        state = state * decay
        key = k[:, token]
        error = v[:, token] - torch.einsum('bhk,bhkv->bhv', key, state)
        state = state + beta[:, token, :, None, None] * key[..., None] * error[..., None, :]
        outputs.append(torch.einsum('bhk,bhkv->bhv', q[:, token] * (q.shape[-1] ** -0.5), state))
    return torch.stack(outputs, dim=1), state

def chunkwise_prefill(variant, inputs, *, state=None):
    q, k, v, g, beta, default_state = inputs
    state = default_state if state is None else state
    if variant == 'GDN':
        return chunk_gated_delta_rule(q, k, v, g, beta, initial_state=state, output_final_state=True)
    return chunk_kda(q, k, v, g, beta, initial_state=state, output_final_state=True)

def sdpa_prefill(inputs):
    q, k, v, *_ = inputs
    return F.scaled_dot_product_attention(
        q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2),
        is_causal=True, dropout_p=0.0,
    ).transpose(1, 2)

## Correctness gate

Do not report timings if either assertion fails. BF16/FP16 inputs permit small rounding differences, but the chunked and recurrent equations must agree with the reference output and final state.

In [ ]:
def verify_variant(variant, tokens=128):
    channelwise = variant == 'KDA'
    inputs = make_inputs(tokens, channelwise=channelwise, batch=2)
    reference_output, reference_state = reference_scan(
        *inputs[:5], channelwise=channelwise, initial_state=inputs[-1]
    )
    with torch.inference_mode():
        chunk_output, chunk_state = chunkwise_prefill(variant, inputs)
    output_error = (chunk_output.float() - reference_output).abs().max().item()
    state_error = (chunk_state.float() - reference_state).abs().max().item()
    torch.testing.assert_close(chunk_output.float(), reference_output, rtol=5e-2, atol=5e-2)
    torch.testing.assert_close(chunk_state.float(), reference_state, rtol=5e-2, atol=5e-2)
    return {'variant': variant, 'max_abs_output_error': output_error, 'max_abs_state_error': state_error}

pd.DataFrame([verify_variant('GDN'), verify_variant('KDA')])

In [ ]:
def p50_ms(operation, *, warmup=10, repeats=30):
    for _ in range(warmup):
        operation()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        torch.cuda.synchronize()
        started = time.perf_counter_ns()
        operation()
        torch.cuda.synchronize()
        samples.append((time.perf_counter_ns() - started) / 1e6)
    samples.sort()
    return {'p50_ms': samples[len(samples) // 2], 'p95_ms': samples[int(0.95 * (len(samples) - 1))]}

def prefill_row(name, tokens, operation, *, batch=SHAPE.batch):
    timing = p50_ms(operation)
    return {
        'operator': name, 'tokens': tokens, 'batch': batch, **timing,
        'tokens_per_second_p50': 1000 * batch * tokens / timing['p50_ms'],
    }

## Prefill throughput

GDN/KDA use `chunk_gated_delta_rule`/`chunk_kda`; MHA uses PyTorch fused SDPA. The Python reference is intentionally measured only at 64 tokens: at 1K/4K it measures launch and interpreter overhead, not a usable kernel.

In [ ]:
PREFILL_LENGTHS = (64, 256, 1024, 4096)
prefill_rows = []
for tokens in PREFILL_LENGTHS:
    mha_inputs = make_inputs(tokens, channelwise=False)
    prefill_rows.append(prefill_row('MHA SDPA', tokens, lambda x=mha_inputs: sdpa_prefill(x)))
    for variant in ('GDN', 'KDA'):
        inputs = make_inputs(tokens, channelwise=variant == 'KDA')
        prefill_rows.append(prefill_row(f'{variant} chunkwise', tokens, lambda v=variant, x=inputs: chunkwise_prefill(v, x)))
        if tokens == 64:
            prefill_rows.append(prefill_row(f'{variant} reference', tokens, lambda x=inputs, c=variant == 'KDA': reference_scan(*x[:5], channelwise=c, initial_state=x[-1])))

prefill_results = pd.DataFrame(prefill_rows)
display(prefill_results.sort_values(['tokens', 'operator']))

## Training-step throughput

This is an attention-operator `forward + backward` microbenchmark at the original 64-token training window. It is intentionally separate from end-to-end GPT training, which also includes embeddings, MLP, layer norms, loss, and AdamW.

In [ ]:
def training_operation(name, inputs):
    def operation():
        q, k, v, g, beta, state = [x.detach().clone() for x in inputs]
        for tensor in (q, k, v, g, beta):
            tensor.requires_grad_(True)
        if name == 'MHA SDPA':
            output = sdpa_prefill((q, k, v, g, beta, state))
        elif name == 'GDN chunkwise':
            output, _ = chunkwise_prefill('GDN', (q, k, v, g, beta, state))
        elif name == 'KDA chunkwise':
            output, _ = chunkwise_prefill('KDA', (q, k, v, g, beta, state))
        else:
            output, _ = reference_scan(q, k, v, g, beta, channelwise=name.startswith('KDA'), initial_state=state)
        output.float().square().mean().backward()
    return operation

TRAIN_TOKENS = 64
training_rows = []
for name, channelwise in (
    ('MHA SDPA', False), ('GDN reference', False), ('GDN chunkwise', False),
    ('KDA reference', True), ('KDA chunkwise', True),
):
    inputs = make_inputs(TRAIN_TOKENS, channelwise=channelwise)
    timing = p50_ms(training_operation(name, inputs), warmup=5, repeats=15)
    training_rows.append({
        'operator': name, 'tokens': TRAIN_TOKENS, 'batch': SHAPE.batch, **timing,
        'tokens_per_second_p50': 1000 * SHAPE.batch * TRAIN_TOKENS / timing['p50_ms'],
    })
training_results = pd.DataFrame(training_rows)
display(training_results.sort_values('p50_ms'))

## Cached one-token decode

The linear paths prefill a fixed FP32 recurrent state with a chunkwise kernel, then use FLA's fused recurrent kernel for each generated token. MHA uses SDPA over a growing K/V cache. Decode is reported at a 4K-token prefix; increase `PROMPT_TOKENS` only after the notebook completes successfully.

In [ ]:
PROMPT_TOKENS, DECODE_TOKENS = 4096, 128

def linear_decode_operation(variant):
    inputs = make_inputs(PROMPT_TOKENS + DECODE_TOKENS, channelwise=variant == 'KDA')
    prefix = tuple(x[:, :PROMPT_TOKENS] if x.ndim >= 2 and x.shape[1] > 1 else x for x in inputs[:-1]) + (inputs[-1],)
    with torch.inference_mode():
        _, prefix_state = chunkwise_prefill(variant, prefix)
    def operation():
        state = prefix_state.clone()
        for index in range(PROMPT_TOKENS, PROMPT_TOKENS + DECODE_TOKENS):
            q, k, v, g, beta = (x[:, index:index + 1] for x in inputs[:5])
            if variant == 'GDN':
                _, state = fused_recurrent_gdn(q, k, v, g, beta, initial_state=state, output_final_state=True)
            else:
                _, state = fused_recurrent_kda(q, k, v, g, beta, initial_state=state, output_final_state=True)
    return operation

def mha_decode_operation():
    q, k, v, *_ = make_inputs(PROMPT_TOKENS + DECODE_TOKENS, channelwise=False)
    def operation():
        for index in range(PROMPT_TOKENS, PROMPT_TOKENS + DECODE_TOKENS):
            query = q[:, index:index + 1].transpose(1, 2)
            keys = k[:, :index].transpose(1, 2)
            values = v[:, :index].transpose(1, 2)
            F.scaled_dot_product_attention(query, keys, values, dropout_p=0.0, is_causal=False)
    return operation

decode_rows = []
for name, operation in (
    ('MHA SDPA', mha_decode_operation()),
    ('GDN fused recurrent', linear_decode_operation('GDN')),
    ('KDA fused recurrent', linear_decode_operation('KDA')),
):
    timing = p50_ms(operation, warmup=5, repeats=15)
    decode_rows.append({
        'operator': name, 'prompt_tokens': PROMPT_TOKENS, 'decode_tokens': DECODE_TOKENS,
        'p50_ms_per_token': timing['p50_ms'] / DECODE_TOKENS,
        'p95_ms_per_token': timing['p95_ms'] / DECODE_TOKENS,
    })
decode_results = pd.DataFrame(decode_rows)
display(decode_results.sort_values('p50_ms_per_token'))

In [ ]:
metadata = pd.DataFrame([{
    'gpu': GPU_NAME, 'compute_capability': '.'.join(map(str, CAPABILITY)),
    'torch': torch.__version__, 'fla': __import__('fla').__version__ if hasattr(__import__('fla'), '__version__') else 'unknown',
    'dtype': str(DTYPE), 'batch': SHAPE.batch, 'heads': SHAPE.heads, 'head_dim': SHAPE.head_dim,
}])
display(metadata)

prefill_results.to_csv('chunkwise_prefill.csv', index=False)
training_results.to_csv('chunkwise_training.csv', index=False)
decode_results.to_csv('chunkwise_decode.csv', index=False)
metadata.to_csv('chunkwise_environment.csv', index=False)
print('Download the four CSV files and place them in the project results/ directory; do not overwrite the CPU reference JSON.')